[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/UoA-eResearch/embedding-llms-qualitative-data-workshop/blob/main/notebooks/02-llms-as-instruments.ipynb)

# Notebook 02 — LLMs as Research Instruments

**Duration:** ~20 minutes

In notebook 01 you connected to a large language model (LLM) and asked it a question. In this notebook we look at the model more carefully: what does it already know, where does its knowledge break down, and why does it sometimes give different answers to the same question?

An LLM is a system trained on a very large amount of text. It learns patterns from that text — facts, grammar, reasoning styles — and uses those patterns to generate new text. It does not look anything up. It does not search the internet. Everything it produces comes from patterns it absorbed during training.

This makes it useful, but also unreliable in specific ways. By the end of this notebook you will understand those limitations well enough to work with them.

## Setup

Run this cell first. Paste your Groq API key between the quotes.

In [ ]:
# ============================================================
# SETUP CELL — Run this once at the start of every notebook
# ============================================================

!pip install groq requests lxml Pillow
import os, json, base64, requests, io
from groq import Groq
from lxml import etree
from PIL import Image
from IPython.display import Image as IPImage, display

os.environ["GROQ_API_KEY"] = "paste_your_key_here"   # <-- paste your key here
client = Groq(api_key=os.environ["GROQ_API_KEY"])
TEXT_MODEL = "llama-3.3-70b-versatile"
VISION_MODEL = "meta-llama/llama-4-maverick-17b-128e-instruct"

print("Setup complete.")

## What does the model already know?

The model was trained on a broad range of text — books, websites, legal documents, academic papers. That means it has **implicit knowledge**: things it absorbed during training, without being given a specific source to read.

This is useful. But it is also risky, because you cannot tell which facts it learned accurately and which ones it reconstructed imperfectly.

Let's test this. We will ask the model about two real New Zealand Acts — without giving it any source text to read.

## Exercise a — A well-known Act

The Privacy Act 2020 is widely discussed in legal and policy writing. The model has almost certainly seen many documents about it during training.

Run the cell below. We are not giving the model any data — just asking what it already knows.

In [ ]:
response = client.chat.completions.create(
    model=TEXT_MODEL,
    messages=[
        {"role": "user", "content": "What are the main purposes of New Zealand's Privacy Act 2020? List them briefly."}
    ]
)

print(response.choices[0].message.content)

You should see a confident, detailed answer. The model sounds like it knows this Act well.

Now let's try something less well-known.

## Exercise b — An obscure Act

The Wool Board Disestablishment Act 2009 dissolved the New Zealand Wool Board. It is a real Act, but it is unlikely to appear in many training documents.

Run the cell below — same question format, different Act.

In [ ]:
response = client.chat.completions.create(
    model=TEXT_MODEL,
    messages=[
        {"role": "user", "content": "What are the main purposes of New Zealand's Wool Board Disestablishment Act 2009? List them briefly."}
    ]
)

print(response.choices[0].message.content)

Compare the two answers:

- Did the model sound equally confident about both Acts?
- Did it give specific, verifiable details about the Wool Board Act, or did it produce vague, plausible-sounding text?
- Would you trust either answer without checking the source?

This is **implicit knowledge** in action. The model does not tell you when it is guessing. It produces fluent text regardless of whether its information is accurate. For well-documented topics it is often right. For obscure ones it may **hallucinate** — generate confident-sounding text that is partially or entirely wrong.

This is why, in notebooks 03–05, we will always give the model **real source data** to work with rather than relying on what it already knows.

## Consistency and temperature

> **Research context:** A published study that computationally analysed New Zealand's Mental Health Act (Ardekani et al., 2026) measured the precision of LLM-based extraction against a deterministic rule-based approach. The LLM achieved 86% precision. The rule-based method achieved 96%. The gap comes from **stochasticity** — the model does not give the same answer every time. What you are about to observe is the same phenomenon the authors measured.

When you send a message to an LLM, the model does not simply look up an answer. It generates text one word at a time, choosing each word based on probabilities. A setting called **temperature** controls how those choices are made:

| Temperature | Behaviour | Use case |
|-------------|-----------|----------|
| `0` | Always picks the most probable word. The output is nearly identical every time. | Research tasks where consistency matters |
| `1` | Picks words with more randomness. The output varies between runs. | Creative writing, brainstorming |

For research, this matters: if you ask an LLM to classify 200 sections of legislation, you need to know whether it would give the same classifications if you ran it again tomorrow.

## Exercise c — Temperature comparison

We will send the **exact same prompt** to the model four times:
- Twice at `temperature=0` (deterministic)
- Twice at `temperature=1` (more random)

Watch for differences between runs.

### Temperature = 0 (first run)

In [ ]:
prompt = "List three key obligations created by New Zealand's Privacy Act 2020. Be brief."

response = client.chat.completions.create(
    model=TEXT_MODEL,
    temperature=0,
    messages=[
        {"role": "user", "content": prompt}
    ]
)

print("=== Temperature 0, Run 1 ===")
print(response.choices[0].message.content)

### Temperature = 0 (second run)

Run this cell. The prompt is identical. Is the answer the same?

In [ ]:
response = client.chat.completions.create(
    model=TEXT_MODEL,
    temperature=0,
    messages=[
        {"role": "user", "content": prompt}
    ]
)

print("=== Temperature 0, Run 2 ===")
print(response.choices[0].message.content)

The two answers should be identical or very close. Now let's see what happens when we increase the temperature.

### Temperature = 1 (first run)

In [ ]:
response = client.chat.completions.create(
    model=TEXT_MODEL,
    temperature=1,
    messages=[
        {"role": "user", "content": prompt}
    ]
)

print("=== Temperature 1, Run 1 ===")
print(response.choices[0].message.content)

### Temperature = 1 (second run)

Same prompt again. Compare with the run above.

In [ ]:
response = client.chat.completions.create(
    model=TEXT_MODEL,
    temperature=1,
    messages=[
        {"role": "user", "content": prompt}
    ]
)

print("=== Temperature 1, Run 2 ===")
print(response.choices[0].message.content)

### What did you notice?

At temperature 0, the two answers should be the same (or nearly). At temperature 1, they probably differ — different wording, different examples, maybe different obligations entirely.

Neither answer is wrong. But they are **inconsistent**. If you used this to classify 200 legislative sections, different runs might produce different classifications for the same section.

The paper measured this precisely: LLM-based extraction achieved **86% precision**, compared to **96% for a rule-based method**. The gap is not because the model is unintelligent — it is because the model is probabilistic. Temperature is the knob that controls how much of that randomness you allow in.

**Rule of thumb for this workshop:** we will use `temperature=0` for all research tasks from notebook 03 onward. Consistency comes first.

## Exercise d — Structuring your prompts

So far we have sent short, informal questions to the model. That works for a quick test, but for research you need more control over what comes back.

A well-structured prompt has four parts:

| Part | Purpose | Example |
|------|---------|--------|
| **Role** | Tell the model who it is | "You are a legal researcher..." |
| **Instruction** | Tell it what to do | "Summarise the following text..." |
| **Context** | Provide the data to work with | The actual text to analyse |
| **Output format** | Specify what you want back | "Respond in one sentence." |

In the Groq API, the **role** is set using a `system` message, and the rest goes in the `user` message. Here is the pattern:

In [ ]:
response = client.chat.completions.create(
    model=TEXT_MODEL,
    temperature=0,
    messages=[
        {
            "role": "system",
            "content": "You are a legal researcher who specialises in New Zealand privacy law."
        },
        {
            "role": "user",
            "content": """Summarise the main change introduced by New Zealand's Privacy Act 2020
compared to the earlier Privacy Act 1993.

Respond in exactly two sentences."""
        }
    ]
)

print(response.choices[0].message.content)

Notice what changed from our earlier calls:

- The `system` message sets the **role**. It shapes the model's perspective for the whole conversation.
- The `user` message combines the **instruction** ("Summarise the main change..."), **context** (the two Acts named), and **output format** ("Respond in exactly two sentences").
- We set `temperature=0` so the answer is reproducible.

This four-part structure is what we will use in every exercise from notebook 03 onward.

### Your turn

Fill in the blanks in the cell below. Pick a role, write an instruction, and specify an output format. Use any topic you like — your own research area, a New Zealand Act, or something completely different.

In [ ]:
# Fill in the blanks below.
#
# Role ideas (pick one or write your own):
#   "You are a public health researcher."
#   "You are a policy analyst working for a New Zealand government ministry."
#   "You are a qualitative researcher who specialises in thematic analysis."
#
# Instruction ideas:
#   "Explain the concept of informed consent in plain English."
#   "List three ethical risks of using AI in social science research."
#   "Compare inductive and deductive coding in qualitative research."
#
# Output format ideas:
#   "Respond in three bullet points."
#   "Respond in one paragraph of no more than 50 words."
#   "Respond as a numbered list."

my_role = "You are a qualitative researcher who specialises in thematic analysis."
my_instruction = "Compare inductive and deductive coding in qualitative research."
my_format = "Respond in three bullet points."

response = client.chat.completions.create(
    model=TEXT_MODEL,
    temperature=0,
    messages=[
        {"role": "system", "content": my_role},
        {"role": "user", "content": my_instruction + "\n\n" + my_format}
    ]
)

print(response.choices[0].message.content)

Try changing the role and running the cell again with the same instruction. Does the answer change? In notebook 04, we will use this deliberately — running the same analysis from different researcher perspectives to see how framing shapes results.

## What you accomplished

In this notebook you:

- Tested the model's **implicit knowledge** and saw that it is confident on well-known topics but unreliable on obscure ones
- Observed **stochasticity** firsthand: the same prompt produces different answers depending on the temperature setting
- Learned that `temperature=0` gives reproducible results — essential for research tasks
- Practised the **four-part prompt structure** (role, instruction, context, output format) that we use for every exercise from here on

The key insight: an LLM is a **research instrument**, not an oracle. Like any instrument, it needs to be calibrated (temperature), pointed at real data (not just implicit knowledge), and validated (which we start doing in notebook 03).

**Next up:** After the break, notebook 03 gives the model real legislative data for the first time. You will compare what a rule-based method finds against what the LLM finds — and measure the gap.